<a href="https://colab.research.google.com/github/Gabriel-Souza18/TP1_mineracao_dados/blob/main/TP1Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install kagglehub -q

import kagglehub

# baixa o dataset e retorna o caminho local já pronto
caminho = kagglehub.dataset_download("mkashifn/nbaiot-dataset")
print("Dataset baixado em:", caminho)

import os
print(os.listdir(caminho)[:10])

Using Colab cache for faster access to the 'nbaiot-dataset' dataset.
Dataset baixado em: /kaggle/input/nbaiot-dataset
['7.gafgyt.combo.csv', '9.gafgyt.combo.csv', '5.gafgyt.combo.csv', '1.mirai.udp.csv', '4.gafgyt.udp.csv', '6.gafgyt.udp.csv', '6.gafgyt.junk.csv', 'data_summary.csv', '5.gafgyt.udp.csv', '9.gafgyt.junk.csv']


In [6]:
import pandas as pd
import glob
import os

arquivos = glob.glob(os.path.join(caminho, "**", "*.csv"), recursive=True)
print(f"Arquivos encontrados: {len(arquivos)}")
for a in arquivos[:]:
    print(os.path.basename(a))

Arquivos encontrados: 92
7.gafgyt.combo.csv
9.gafgyt.combo.csv
5.gafgyt.combo.csv
1.mirai.udp.csv
4.gafgyt.udp.csv
6.gafgyt.udp.csv
6.gafgyt.junk.csv
data_summary.csv
5.gafgyt.udp.csv
9.gafgyt.junk.csv
9.mirai.scan.csv
1.benign.csv
2.mirai.udpplain.csv
3.gafgyt.combo.csv
4.gafgyt.combo.csv
6.mirai.scan.csv
5.mirai.udp.csv
3.benign.csv
3.gafgyt.junk.csv
7.gafgyt.scan.csv
features.csv
6.mirai.udp.csv
6.gafgyt.tcp.csv
6.gafgyt.combo.csv
6.mirai.syn.csv
7.gafgyt.udp.csv
5.gafgyt.junk.csv
8.gafgyt.combo.csv
7.gafgyt.junk.csv
5.mirai.ack.csv
4.mirai.syn.csv
2.gafgyt.tcp.csv
6.mirai.ack.csv
2.mirai.ack.csv
1.gafgyt.combo.csv
6.mirai.udpplain.csv
4.benign.csv
2.gafgyt.scan.csv
6.benign.csv
4.mirai.udp.csv
9.mirai.syn.csv
9.mirai.udp.csv
device_info.csv
2.gafgyt.combo.csv
5.gafgyt.scan.csv
6.gafgyt.scan.csv
9.mirai.udpplain.csv
1.gafgyt.udp.csv
9.gafgyt.scan.csv
2.mirai.scan.csv
8.gafgyt.scan.csv
5.gafgyt.tcp.csv
5.mirai.scan.csv
5.benign.csv
4.mirai.scan.csv
9.benign.csv
2.gafgyt.udp.csv
1.mir

In [5]:
import pandas as pd
import os
import gc

arquivos_auxiliares = {"data_summary.csv", "features.csv"}
N_AMOSTRA_POR_ARQUIVO = 5000  #Colab nao aguentou sem isso

dfs = []
arquivos_ignorados = []

for arq in arquivos:
    nome_arquivo = os.path.basename(arq)
    if nome_arquivo in arquivos_auxiliares:
        continue

    nome = nome_arquivo.replace(".csv", "")
    partes = nome.split(".")
    dispositivo = partes[0]

    if "benign" in nome:
        classe_binaria, familia_ataque, tipo_ataque = "benign", "benign", "benign"
    elif len(partes) >= 3:
        classe_binaria = "attack"
        familia_ataque = partes[1]
        tipo_ataque = ".".join(partes[2:])
    else:
        arquivos_ignorados.append(nome_arquivo)
        continue

    # lê e já converte pra float32 (economizar RAM)
    df_temp = pd.read_csv(arq, dtype="float32")

    if len(df_temp) > N_AMOSTRA_POR_ARQUIVO:
        df_temp = df_temp.sample(n=N_AMOSTRA_POR_ARQUIVO, random_state=42)

    df_temp["dispositivo"] = dispositivo
    df_temp["classe_binaria"] = classe_binaria
    df_temp["familia_ataque"] = familia_ataque
    df_temp["tipo_ataque"] = tipo_ataque

    dfs.append(df_temp)
    del df_temp
    gc.collect()

df = pd.concat(dfs, ignore_index=True)
del dfs
gc.collect()

print(f"Total de linhas: {df.shape[0]}, colunas: {df.shape[1]}")
print(df[["dispositivo", "classe_binaria", "familia_ataque", "tipo_ataque"]].value_counts())

Total de linhas: 445000, colunas: 119
dispositivo  classe_binaria  familia_ataque  tipo_ataque
1            attack          gafgyt          combo          5000
                                             junk           5000
                                             scan           5000
                                             tcp            5000
                                             udp            5000
                                                            ... 
9            attack          mirai           scan           5000
                                             syn            5000
                                             udp            5000
                                             udpplain       5000
             benign          benign          benign         5000
Name: count, Length: 89, dtype: int64


# Pré-processamento